In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/home-credit-default-risk/sample_submission.csv
/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv
/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv
/kaggle/input/competitions/home-credit-default-risk/application_train.csv
/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv
/kaggle/input/competitions/home-credit-default-risk/application_test.csv
/kaggle/input/competitions/home-credit-default-risk/previous_application.csv
/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv
/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv
/kaggle/input/competitions/home-credit-default-risk/bureau.csv


In [2]:
import pandas as pd
homecredit_columns_description = pd.read_csv(
    '/kaggle/input/competitions/home-credit-default-risk/HomeCredit_columns_description.csv', 
    encoding='latin1')
pos_cash_balance = pd.read_csv(
    '/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv')
app_train = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_train.csv')
app_test  = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/application_test.csv')
bureau         = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau.csv')
bureau_balance = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/bureau_balance.csv')
credit_card    = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv')
installments   = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv')
previous_app   = pd.read_csv('/kaggle/input/competitions/home-credit-default-risk/previous_application.csv')
print("All files loaded successfully!")
print(f"Training set shape: {app_train.shape}")


All files loaded successfully!
Training set shape: (307511, 122)


In [3]:
tables={'homecredit_columns_description':homecredit_columns_description,'pos_cash_balance':pos_cash_balance,'app_train':app_train,
       'app_test':app_test,'bureau':bureau,'bureau_balance':bureau_balance,'credit_card':credit_card,'installments':installments,
       'previous_app':previous_app}

In [4]:
for name,table in tables.items():
    print(name,table.shape)

homecredit_columns_description (219, 5)
pos_cash_balance (10001358, 8)
app_train (307511, 122)
app_test (48744, 121)
bureau (1716428, 17)
bureau_balance (27299925, 3)
credit_card (3840312, 23)
installments (13605401, 8)
previous_app (1670214, 37)


In [5]:
# for name,table in tables.items():
#     for column in table:
#         print(name,column,table[column].isna().sum(),(table[column].isna().sum()/len(table))*100.0)

In [6]:
import duckdb as db
test1=db.query("""
with compressed_pos_cash_balance as(
select distinct sk_id_curr,max(cnt_instalment) as max_instalment
from pos_cash_balance
group by sk_id_curr
),
main_table as(
select case when p.max_instalment is null then 'no pos history' when p.max_instalment > 36.0 and p.max_instalment <= 48.0 then '4th year' 
when p.max_instalment > 24.0 and p.max_instalment <= 36.0 then '3rd year' 
when p.max_instalment > 12.0 and p.max_instalment <= 24.0 then '2nd year' 
when p.max_instalment > 0.0  and p.max_instalment <= 12.0 then '1st year' 
when p.max_instalment > 48.0 then '4+ years' else 'unknown' end as cnt_instalment_year,a.amt_income_total,a.amt_credit,a.amt_annuity,a.target
from app_train a 
left join compressed_pos_cash_balance p on p.sk_id_curr=a.sk_id_curr
)
select 
cnt_instalment_year,
count(*) as total_unique_customers,
round(avg(amt_credit), 2) as avg_credit_amount,
round(avg(amt_income_total), 2) as avg_income,
round(avg(amt_annuity), 2) as avg_amount_pay,
sum(target) as total_defaulted_customers,
round(sum(target) * 100.0 / count(*), 2) as default_rate_percentage
from main_table
group by cnt_instalment_year
order by total_unique_customers desc
""").df()
test1.head(10)

,cnt_instalment_year,total_unique_customers,avg_credit_amount,avg_income,avg_amount_pay,total_defaulted_customers,default_rate_percentage
0,1st year,148313,578320.19,163432.78,27241.30,11951.0,8.06
1,2nd year,78851,590413.30,165674.05,27031.16,5975.0,7.58
2,3rd year,30858,623260.51,175742.71,27120.16,2631.0,8.53
3,4th year,18441,621777.00,179043.38,26201.63,1940.0,10.52
4,no pos history,18091,725478.43,196729.05,28767.36,1209.0,6.68
5,4+ years,12957,621795.90,179101.08,25007.53,1119.0,8.64


In [7]:
import duckdb as db
print(db.query("""
select
  count(*) as applicants,
  sum(case when days_employed = 365243 then 1 else 0 end) as sentinel_rows,
  round(avg(case when days_employed = 365243 then 1.0 else 0 end)*100, 2) as sentinel_pct,
  round(avg(case when days_employed = 365243 then target end)*100, 2) as default_pct_sentinel,
  round(avg(case when days_employed <> 365243 then target end)*100, 2) as default_pct_other
from app_train
""").df())

print(db.query("select code_gender, count(*) as n from app_train group by 1 order by n desc").df())
print(db.query("select count(*) as xna_org from app_train where organization_type = 'XNA'").df())

   applicants  sentinel_rows  sentinel_pct  default_pct_sentinel  \
0      307511        55374.0         18.01                   5.4   

   default_pct_other  
0               8.66  
  CODE_GENDER       n
0           F  202448
1           M  105059
2         XNA       4
   xna_org
0    55374


In [8]:
test2=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
x_flag,
case when (c_flag>0.0 or flag_0>0.0) and greatest(flag_1,flag_2,flag_3,flag_4,flag_5,x_flag)=0.0 then 'safe' 
when (c_flag>0) and greatest(flag_1,flag_2,flag_3)>0.0 and greatest(flag_4,flag_5)=0.0 then 'closed after delays'
when (c_flag =0.0) and greatest(flag_1,flag_2,flag_3)>0.0 and greatest(flag_4,flag_5)=0.0 then '1-3 months delay'
when (c_flag>0) and greatest(flag_4,flag_5)>0 then 'closed with heavy delays'
when (c_flag =0.0) and greatest(flag_4,flag_5)>0.0 then '3> months delay' 
else 'unclassified' end as flags
from main_table
)
select count(*)
from calculation_table
where x_flag>0 and flags='unclassified'
""").df()
test2.head(10)

,count_star()
0,398776


In [9]:
exp1=db.query("""
select  status,count(status)
from bureau_balance
group by status
""").df()
exp1.head(10)

,STATUS,count(status)
0,1,242347
1,5,62406
2,2,23419
3,C,13646993
4,0,7499507
5,X,5810482
6,4,5847
7,3,8924


In [10]:
test3=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
case when greatest(flag_1,flag_2,flag_3,flag_4,flag_5)=0 then 'none' 
when greatest(flag_1,flag_2,flag_3)>0 and greatest(flag_4,flag_5)=0 then 'mild'
when greatest(flag_4,flag_5)>0 then 'heavy' end as delay,
case when x_flag>0 then 'yes' else 'no' end as has_x,
case when c_flag>0 then 'yes' else 'no' end as has_c
from main_table
)
select delay,has_x,has_c,count(*) as count
from calculation_table
group by delay,has_x,has_c
""").df()
print(test3.shape)

(12, 4)


In [11]:
test4=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
case when greatest(flag_1,flag_2,flag_3,flag_4,flag_5)=0 then 'none' 
when greatest(flag_1,flag_2,flag_3)>0 and greatest(flag_4,flag_5)=0 then 'mild'
when greatest(flag_4,flag_5)>0 then 'heavy' end as delay,
case when x_flag>0 then 'yes' else 'no' end as has_x,
case when c_flag>0 then 'yes' else 'no' end as has_c
from main_table
),
joined_table as (
select b.sk_id_curr,c.delay,c.has_x,c.has_c
from calculation_table c 
left join bureau b on b.sk_id_bureau=c.sk_id_bureau
),
joined_table2 as(
select j.sk_id_curr,max(case when j.delay='none' then 0 when j.delay='mild' then 1 when j.delay='heavy' then 2 else 'no_history'  end) as worst_number
from joined_table j 
inner join app_train a on a.sk_id_curr=j.sk_id_curr
group by j.sk_id_curr
),
total_table as(
select count(*) as total_count
from joined_table2
)
select case when worst_number =0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' end as delay,count(*) as count,total_count,
round((count(*)*100.0/total_count),2) as pct
from joined_table2,total_table
group by case when worst_number =0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' end,total_count
""").df()
test4.head()

,delay,count,total_count,pct
0,heavy,2611,92231,2.83
1,none,61179,92231,66.33
2,mild,28441,92231,30.84


In [12]:
test4b=db.query("""
with main_table as(
select sk_id_bureau,sum(case when status='C' then 1 else 0 end) as c_flag,
sum(case when status='X' then 1 else 0 end) as x_flag,
sum(case when status='0' then 1 else 0 end) as flag_0,
sum(case when status='1' then 1 else 0 end) as flag_1,
sum(case when status='2' then 1 else 0 end) as flag_2,
sum(case when status='3' then 1 else 0 end) as flag_3,
sum(case when status='4' then 1 else 0 end) as flag_4,
sum(case when status='5' then 1 else 0 end) as flag_5
from bureau_balance
group by sk_id_bureau
),
calculation_table as(
select sk_id_bureau,
case when greatest(flag_1,flag_2,flag_3,flag_4,flag_5)=0 then 'none' 
when greatest(flag_1,flag_2,flag_3)>0 and greatest(flag_4,flag_5)=0 then 'mild'
when greatest(flag_4,flag_5)>0 then 'heavy' end as delay,
case when x_flag>0 then 'yes' else 'no' end as has_x,
case when c_flag>0 then 'yes' else 'no' end as has_c
from main_table
),
joined_table as (
select b.sk_id_curr,c.delay,c.has_x,c.has_c
from calculation_table c 
left join bureau b on b.sk_id_bureau=c.sk_id_bureau
),
grouped_table as(
select sk_id_curr,max(case when delay='none' then 0 when delay='mild' then 1 when delay='heavy' then 2 else 3 end) as worst_number
from joined_table
group by sk_id_curr
),
joined_table2 as(
select a.sk_id_curr,g.worst_number,a.target
from app_train a
left join grouped_table g on a.sk_id_curr=g.sk_id_curr
),
calculated_table2 as(
select case when worst_number=0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' else 'no_history' end as delay,
count(*) as total_count_applicants,sum(target) as defaulted_count
from joined_table2
group by case when worst_number=0 then 'none' when worst_number=1 then 'mild' when worst_number=2 then 'heavy' else 'no_history' end
)
select delay,defaulted_count,round((defaulted_count*100.0/total_count_applicants),2) as pct,
case when (defaulted_count*100.0/total_count_applicants)>8.07 then 'yes' else 'no' end as is_greater_than_default
from calculated_table2
""").df()
test4b.head()

,delay,defaulted_count,pct,is_greater_than_default
0,mild,2810.0,9.88,yes
1,no_history,17315.0,8.04,no
2,none,4418.0,7.22,no
3,heavy,282.0,10.80,yes


In [13]:
test5=db.query("""
with main_table as(
select code_gender,name_contract_type,count(*) as applicants,sum(target) as defaulted_count
from app_train
where code_gender!='XNA'
group by code_gender,name_contract_type
)
select code_gender,name_contract_type,defaulted_count,applicants,round((defaulted_count*100.0/applicants),2) as pct
from main_table
order by pct desc
""").df()
test5.head()

,CODE_GENDER,NAME_CONTRACT_TYPE,defaulted_count,applicants,pct
0,M,Cash loans,10094.0,95432,10.58
1,F,Cash loans,13127.0,182800,7.18
2,M,Revolving loans,561.0,9627,5.83
3,F,Revolving loans,1043.0,19648,5.31


In [14]:
test6=db.query("""
with credit_amt_greater_than_amt_income as(
select count(sk_id_curr) as credit_amt_greater_than_income
from app_train
where amt_credit>amt_income_total
),
total as(
select count(sk_id_curr) as total_applicants
from app_train
)
select credit_amt_greater_than_income,total_applicants,
round(((credit_amt_greater_than_income*100.0)/total_applicants),2) as pct
from credit_amt_greater_than_amt_income,total
""").df()
test6.head()

,credit_amt_greater_than_income,total_applicants,pct
0,291337,307511,94.74


In [15]:
app_train.head(1)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0


In [16]:
test7=db.query("""
with main_table as(
select sk_id_curr,amt_annuity,amt_income_total,round((amt_annuity*100.0/amt_income_total),2) as annuity_to_income_pct,target
from app_train
),
calculated_table as(
select case when annuity_to_income_pct<=20 then '0%-20%' 
when annuity_to_income_pct>20 and annuity_to_income_pct<=40 then 'over 20%-40%'
when annuity_to_income_pct>40 then 'over 40%' else 'annuity missing' end as annuity_bucket,
target
from main_table
)
select annuity_bucket,count(*) as count,sum(target),(sum(target)*100.0/count(annuity_bucket)) as default_pct,
case when (sum(target)*100.0/count(annuity_bucket))>8.07 then 'yes' else 'no' end as is_greater_than_default
from calculated_table
group by annuity_bucket
""").df()
test7.head()

,annuity_bucket,count,sum(target),default_pct,is_greater_than_default
0,over 20%-40%,97129,8364.0,8.611228,yes
1,over 40%,7907,654.0,8.271152,yes
2,annuity missing,12,0.0,0.000000,no
3,0%-20%,202463,15807.0,7.807352,no


In [17]:
previous_app.columns

Index(['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY',
       'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE',
       'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
       'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY',
       'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY',
       'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE',
       'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE',
       'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE',
       'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE',
       'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY',
       'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION',
       'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
       'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL'],
      dtype='object')

In [18]:
previous_app.shape

(1670214, 37)

In [19]:
exp111=db.query("""
select name_contract_status,count(*) as count
from previous_app
group by name_contract_status
""").df()
exp111.head()

,NAME_CONTRACT_STATUS,count
0,Approved,1036781
1,Canceled,316319
2,Unused offer,26436
3,Refused,290678


In [20]:
test8=db.query("""
with compressed_previous_app as(
select sk_id_curr,sum(case when name_contract_status='Approved' then 1 else 0 end) as approved_count,
sum(case when name_contract_status='Refused' then 1 else 0 end) as refused_count,
sum(case when name_contract_status='Unused offer' then 1 else 0 end) as unused_offer_count,
sum(case when name_contract_status='Canceled' then 1 else 0 end) as canceled_count
from previous_app
group by sk_id_curr
),
main_table as(
select a.sk_id_curr as present_id,
c.sk_id_curr as prev_id,
c.approved_count,
c.refused_count,
a.target,
c.unused_offer_count,
c.canceled_count
from app_train a 
left join compressed_previous_app c on c.sk_id_curr=a.sk_id_curr
),
calculated_table as(
select case when prev_id is null then 'no_prev_application'
when refused_count>0 then 'at least one refusal'
when approved_count>0 then 'no prev refusal'
when unused_offer_count>0 and refused_count=0 and approved_count=0  then 'unused_offer'
when canceled_count>0 and approved_count=0 and refused_count=0  then 'canceled_offer'
else 'mixed history' end as status,target
from main_table
)
select status,
sum(target) as default_count,
count(*) as total_count,
round((sum(target)*100.0)/count(*),2) as pct,
case when round((sum(target)*100.0)/count(*),2)>8.07 then 'yes' else 'no' end as 'whether above general default',
round((sum(target)*100.0)/count(*),2)-8.07 as diff
from calculated_table
group by status
""").df()
test8.head(10)

,status,default_count,total_count,pct,whether above general default,diff
0,no prev refusal,13472.0,190370,7.08,no,-0.99
1,no_prev_application,980.0,16454,5.96,no,-2.11
2,unused_offer,13.0,166,7.83,no,-0.24
3,at least one refusal,10352.0,100294,10.32,yes,2.25
4,canceled_offer,8.0,227,3.52,no,-4.55


In [21]:
installments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [22]:
previous_app.columns

Index(['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY',
       'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE',
       'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
       'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY',
       'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY',
       'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE',
       'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE',
       'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE',
       'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE',
       'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY',
       'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION',
       'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
       'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL'],
      dtype='object')

In [23]:
exp11111=db.query("""
select name_yield_group,count(*) as count
from previous_app
group by name_yield_group
""").df()
exp11111.head()

,NAME_YIELD_GROUP,count
0,XNA,517215
1,low_normal,322095
2,middle,385532
3,low_action,92041
4,high,353331


In [24]:
exp12345=db.query("""
select name_seller_industry,count(*) as count
from previous_app
group by name_seller_industry
""").df()
exp12345.head()

,NAME_SELLER_INDUSTRY,count
0,Consumer electronics,398265
1,Industry,19194
2,Clothing,23949
3,XNA,855720
4,Connectivity,276029


In [25]:
credit_card.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [26]:
credit_card.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [27]:
test9=db.query("""
with main_table as(
select sk_id_curr,amt_balance,AMT_RECEIVABLE_PRINCIPAL,	round((AMT_RECEIVABLE_PRINCIPAL*100.0/AMT_credit_limit_actual),2) as pct_amt_used_from_total,
case when (AMT_BALANCE!=0 and AMT_RECEIVABLE_PRINCIPAL!=0) and ((AMT_RECEIVABLE_PRINCIPAL*100.0/AMT_credit_limit_actual))>90  then 1 else 0 end as greater_than_pct90_ornot,
count(*) over(partition by sk_id_curr) as count_by_id
from credit_card
),
calculated_table as(
select sk_id_curr,AMT_BALANCE,AMT_RECEIVABLE_PRINCIPAL,pct_amt_used_from_total,greater_than_pct90_ornot,count_by_id
from main_table
),
calculated_table2 as(
select sk_id_curr,avg(pct_amt_used_from_total) as avg_amt_used_from_balance,sum(greater_than_pct90_ornot) as total_count_used_above_pct90,
max(count_by_id) as total_month_active
from calculated_table
group by sk_id_curr
),
calculated_table3 as(
select sk_id_curr,avg_amt_used_from_balance,total_count_used_above_pct90,total_month_active,
case when total_count_used_above_pct90 is null then 'no previous history'
when total_count_used_above_pct90=0 then 'category 0' 
when total_count_used_above_pct90<=10 then 'category 1'
when total_count_used_above_pct90>10 and total_count_used_above_pct90<=20 then 'category 2'
when total_count_used_above_pct90>20 and total_count_used_above_pct90<=30 then 'category 3'
when total_count_used_above_pct90>30 and total_count_used_above_pct90<=40 then 'category 4'
when total_count_used_above_pct90>40 then 'category 5'  end as category
from calculated_table2
),
calculated_table4 as(
select c3.sk_id_curr,c3.avg_amt_used_from_balance,c3.total_count_used_above_pct90,c3.category,a.target,c3.total_month_active
from app_train a
left join calculated_table3 c3 on a.sk_id_curr=c3.sk_id_curr
)
select coalesce(category,'no previous history') as category,sum(target) as count_of_defaults,count(*) as total_count,round((sum(target)*100.0/count(*)),2) as default_pct,round((sum(target)*100.0/count(*)),2)-8.07 as diff,
round(avg(total_month_active),2) as avg_card_months
from calculated_table4
group by category
order by default_pct desc
""").df()
test9.head(10)

,category,count_of_defaults,total_count,default_pct,diff,avg_card_months
0,category 1,3465.0,28500,12.16,4.09,38.62
1,category 2,854.0,7928,10.77,2.70,56.93
2,category 3,338.0,3620,9.34,1.27,70.63
3,category 5,189.0,2221,8.51,0.44,87.09
4,category 4,149.0,1880,7.93,-0.14,78.85
5,no previous history,17291.0,220606,7.84,-0.23,NaN
6,category 0,2539.0,42756,5.94,-2.13,25.22


In [28]:
app_train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
test10=db.query("""
select case when flag_own_car='N' and flag_own_realty='N' then 'no car and house'
when flag_own_car='Y' and flag_own_realty='N' then 'yes car no house'
when flag_own_car='N' and flag_own_realty='Y' then 'no car yes house'
when flag_own_car='Y' and flag_own_realty='Y' then 'yes car and yes house'
end as category,sum(target) as default_count,count(*) as total_count,round((sum(target)*100.0/count(*)),2) as pct,round((sum(target)*100.0/count(*)),2)-8.07 as diff,round(avg(amt_income_total),2) as avg_income
from app_train
group by 1
order by pct desc
""").df()
test10.head()

,category,default_count,total_count,pct,diff,avg_income
0,no car and house,5572.0,61972,8.99,0.92,152472.56
1,no car yes house,11677.0,140952,8.28,0.21,155540.16
2,yes car and yes house,5306.0,72360,7.33,-0.74,195986.07
3,yes car no house,2270.0,32227,7.04,-1.03,197130.97


In [33]:
# homecredit_columns_description.to_csv('homecredit_columns_description.csv',index=False)
# pos_cash_balance.to_csv('pos_cash_balance.csv',index=False)
# app_train.to_csv('app_train.csv',index=False)
# app_test.to_csv('app_test.csv',index=False)
# bureau.to_csv('bureau.csv',index=False)
# bureau_balance.to_csv('bureau_balance.csv',index=False)
# credit_card.to_csv('credit_card.csv',index=False)
# installments.to_csv('installments.csv',index=False)
# previous_app.to_csv('previous_app.csv',index=False)